# 03 — Feature Engineering

This notebook creates additional fraud-risk features from the cleaned BAF dataset.

Only deterministic, row-level transformations are created at this stage. No target-based statistics, imputation, scaling, or learned transformations are applied, so the temporal train/validation/test separation remains leakage-safe.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path(
    r"D:\bank-account-fraud-decision-system"
)

RAW_DATA_DIR = PROJECT_ROOT / "data" / "raw"

df = pd.read_csv(
    RAW_DATA_DIR / "Base.csv"
)

df_clean = df.drop(
    columns=["device_fraud_count"]
).copy()

sentinel_features = [
    "prev_address_months_count",
    "current_address_months_count",
    "bank_months_count",
    "session_length_in_minutes",
    "device_distinct_emails_8w"
]

for col in sentinel_features:
    df_clean[f"{col}_missing"] = (
        df_clean[col] == -1
    ).astype(int)

    df_clean.loc[
        df_clean[col] == -1,
        col
    ] = np.nan

print(df_clean.shape)

(1000000, 36)


## 1. Missing-Information Burden

Previous analysis showed that missingness itself can be predictive of fraud. A combined missing-information count is therefore created to summarize how many sentinel-coded fields are unavailable for each application.

In [2]:
missing_flag_cols = [
    f"{col}_missing"
    for col in sentinel_features
]

df_clean["missing_info_count"] = (
    df_clean[missing_flag_cols]
    .sum(axis=1)
)

print(
    df_clean["missing_info_count"]
    .value_counts()
    .sort_index()
)

missing_info_count
0    194941
1    638136
2    165852
3       941
4       130
Name: count, dtype: int64


## 2. Contact Validity Features

Home-phone and mobile-phone validity are combined into simple contactability features. These features summarize whether an application provides zero, one, or two valid phone contacts.

In [3]:
df_clean["valid_phone_count"] = (
    df_clean["phone_home_valid"]
    + df_clean["phone_mobile_valid"]
)

df_clean["has_any_valid_phone"] = (
    df_clean["valid_phone_count"] > 0
).astype(int)

print(
    df_clean["valid_phone_count"]
    .value_counts()
    .sort_index()
)

valid_phone_count
0     22232
1    648783
2    328985
Name: count, dtype: int64


## 3. Address-History Availability

Previous-address missingness showed a strong association with fraud during EDA. A compact indicator is created to distinguish applications with both address-history fields available from those with incomplete address information.

In [4]:
df_clean["address_history_complete"] = (
    (
        df_clean["prev_address_months_count"].notna()
    )
    &
    (
        df_clean["current_address_months_count"].notna()
    )
).astype(int)

print(
    df_clean["address_history_complete"]
    .value_counts()
)

address_history_complete
0    717088
1    282912
Name: count, dtype: int64


In [5]:
print(
    "Feature-engineered shape:",
    df_clean.shape
)

new_features = [
    "missing_info_count",
    "valid_phone_count",
    "has_any_valid_phone",
    "address_history_complete"
]

display(
    df_clean[new_features].head()
)

Feature-engineered shape: (1000000, 40)


,missing_info_count,valid_phone_count,has_any_valid_phone,address_history_complete
0,1,1,1,0
1,1,2,1,0
2,0,1,1,1
3,0,1,1,1
4,1,2,1,0


## 4. Engineered Feature Signal Check

The newly created features are evaluated against the fraud target to confirm whether they provide useful separation before being retained for modeling.

In [6]:
engineered_features = [
    "missing_info_count",
    "valid_phone_count",
    "has_any_valid_phone",
    "address_history_complete"
]

for col in engineered_features:
    print(f"\n{col}")

    result = (
        df_clean
        .groupby(col)["fraud_bool"]
        .agg(
            applications="count",
            fraud_cases="sum",
            fraud_rate="mean"
        )
    )

    result["fraud_rate_pct"] = (
        result["fraud_rate"] * 100
    ).round(4)

    display(result)


missing_info_count


,applications,fraud_cases,fraud_rate,fraud_rate_pct
missing_info_count,,,,
0,194941,472,0.002421,0.2421
1,638136,6818,0.010684,1.0684
2,165852,3726,0.022466,2.2466
3,941,12,0.012752,1.2752
4,130,1,0.007692,0.7692



valid_phone_count


,applications,fraud_cases,fraud_rate,fraud_rate_pct
valid_phone_count,,,,
0,22232,619,0.027843,2.7843
1,648783,8648,0.013330,1.3330
2,328985,1762,0.005356,0.5356



has_any_valid_phone


,applications,fraud_cases,fraud_rate,fraud_rate_pct
has_any_valid_phone,,,,
0,22232,619,0.027843,2.7843
1,977768,10410,0.010647,1.0647



address_history_complete


,applications,fraud_cases,fraud_rate,fraud_rate_pct
address_history_complete,,,,
0,717088,10148,0.014152,1.4152
1,282912,881,0.003114,0.3114


In [7]:
display(
    df_clean[
        engineered_features + ["fraud_bool"]
    ]
    .corr(numeric_only=True)["fraud_bool"]
    .drop("fraud_bool")
    .sort_values(
        key=abs,
        ascending=False
    )
)

missing_info_count          0.056686
address_history_complete   -0.047602
valid_phone_count          -0.042301
has_any_valid_phone        -0.024276
Name: fraud_bool, dtype: float64

## 5. Additional Risk Interaction Features

A small number of interpretable interaction features are created to summarize potentially higher-risk combinations observed during exploratory analysis.

In [8]:
# Email / identity consistency risk
df_clean["low_identity_consistency"] = (
    df_clean["name_email_similarity"] < 0.4
).astype(int)

# Elevated requested credit relative to typical applicant scale
df_clean["high_credit_request"] = (
    df_clean["proposed_credit_limit"] >= 1000
).astype(int)

# Combined contact-risk indicator
df_clean["contact_risk_flag"] = (
    (
        df_clean["phone_home_valid"] == 0
    )
    &
    (
        df_clean["phone_mobile_valid"] == 0
    )
).astype(int)

In [9]:
new_risk_features = [
    "low_identity_consistency",
    "high_credit_request",
    "contact_risk_flag"
]

for col in new_risk_features:
    print(f"\n{col}")

    result = (
        df_clean
        .groupby(col)["fraud_bool"]
        .agg(
            applications="count",
            fraud_cases="sum",
            fraud_rate="mean"
        )
    )

    result["fraud_rate_pct"] = (
        result["fraud_rate"] * 100
    ).round(4)

    display(result)


low_identity_consistency


,applications,fraud_cases,fraud_rate,fraud_rate_pct
low_identity_consistency,,,,
0,591740,4611,0.007792,0.7792
1,408260,6418,0.015720,1.5720



high_credit_request


,applications,fraud_cases,fraud_rate,fraud_rate_pct
high_credit_request,,,,
0,765838,6064,0.007918,0.7918
1,234162,4965,0.021203,2.1203



contact_risk_flag


,applications,fraud_cases,fraud_rate,fraud_rate_pct
contact_risk_flag,,,,
0,977768,10410,0.010647,1.0647
1,22232,619,0.027843,2.7843


## 6. Final Feature Selection

Engineered features were retained only when they showed clear fraud-rate separation and had a defensible operational interpretation.

Redundant binary contact indicators are excluded in favor of `valid_phone_count`, which preserves more information. Heuristic risk flags are retained as interpretable interaction features but are not treated as optimized decision thresholds.

In [10]:
df_model = df_clean.drop(
    columns=[
        "has_any_valid_phone",
        "contact_risk_flag"
    ]
).copy()

print("Final modeling dataset shape:", df_model.shape)

Final modeling dataset shape: (1000000, 41)


In [11]:
print(df_model["fraud_bool"].value_counts())

print(
    df_model["month"]
    .value_counts()
    .sort_index()
)

fraud_bool
0    988971
1     11029
Name: count, dtype: int64
month
0    132440
1    127620
2    136979
3    150936
4    127691
5    119323
6    108168
7     96843
Name: count, dtype: int64


In [12]:
import pyarrow
print(pyarrow.__version__)



21.0.0


In [13]:
PROCESSED_DATA_DIR = PROJECT_ROOT / "data" / "processed"

PROCESSED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

output_path = (
    PROCESSED_DATA_DIR
    / "fraud_modeling_dataset.parquet"
)

df_model.to_parquet(
    output_path,
    index=False
)

print("Saved to:", output_path)

Saved to: D:\bank-account-fraud-decision-system\data\processed\fraud_modeling_dataset.parquet


## 7. Feature Engineering Summary

The feature-engineering stage produced a modeling dataset containing the original application features, explicit missingness indicators, and a small set of interpretable fraud-risk features.

Engineered variables capture overall missing-information burden, contact validity, address-history completeness, identity consistency, and elevated requested credit. Each retained feature demonstrated measurable fraud-rate separation during validation.

No imputation, scaling, target encoding, or model-derived transformation was performed at this stage. These operations will be fitted only on historical training data to preserve the chronological evaluation design.